In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.get("https://www.samsung.com/in/smartphones/all-smartphones/")

wait = WebDriverWait(driver, 10)

while True:
    try:
        # Button tak scroll karo
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

        view_more = wait.until(
            EC.element_to_be_clickable(
                (By.CSS_SELECTOR, ".pd21-product-finder__view-more")
            )
        )

        # JavaScript click zyada reliable hota hai
        driver.execute_script("arguments[0].click();", view_more)

        print("Clicked View More")
        time.sleep(3)

    except Exception:
        print("No more products to load.")
        break

Clicked View More
Clicked View More
Clicked View More
Clicked View More
Clicked View More
Clicked View More
Clicked View More
Clicked View More
Clicked View More
No more products to load.


In [3]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(driver.page_source, "lxml")

cards = soup.select(".pd21-product-card__item.js-pfv2-product-card")

print("Total cards:", len(cards))

Total cards: 104


In [4]:
data = []

for card in cards:
    name = card.select_one(".pd21-product-card__name")

    if not name:  # skip non-product cards
        continue

    price = card.select_one(".price-ux__price-current strong")
    original_price = card.select_one(".price-ux__price-original")
    rating = card.select_one(".rating__point span:last-child")
    reviews = card.select_one(".rating__review-count span:last-child")
    color = card.select_one(".option-selector-v2__color-name--pc")
    storage = card.select_one(
        ".option-selector-v2__swiper-slide.is-checked .option-selector-v2__size-text"
    )

    data.append({
        "name": name.get_text(strip=True),
        "price": price.get_text(strip=True) if price else None,
        "original_price": original_price.get_text(strip=True) if original_price else None,
        "rating": rating.get_text(strip=True) if rating else None,
        "reviews": reviews.get_text(strip=True) if reviews else None,
        "color": color.get_text(strip=True) if color else None,
        "storage": storage.get_text(strip=True) if storage else None,
    })

In [5]:
df = pd.DataFrame(data)

print(df)

                                  name       price  \
0                     Galaxy S26 Ultra  ₹130999.00   
1                          Galaxy S26+   ₹94999.00   
2                           Galaxy S26   ₹99999.00   
3    Galaxy S26 Ultra (Special Colour)  ₹150999.00   
4         Galaxy S26+ (Special Colour)   ₹94999.00   
..                                 ...         ...   
99         Galaxy F15 5G (4 GB Memory)        None   
100        Galaxy A15 5G (6 GB Memory)        None   
101                   Galaxy S23 Ultra        None   
102        Galaxy A15 5G (8 GB Memory)        None   
103           Galaxy F13 (4 GB Memory)        None   

                           original_price rating reviews           color  \
0    MRP: ₹139999.00 (Incl. of all taxes)    4.8   9,150   Cobalt Violet   
1    MRP: ₹119999.00 (Incl. of all taxes)    4.8   1,918           Black   
2    MRP: ₹107999.00 (Incl. of all taxes)    4.8   2,231   Cobalt Violet   
3    MRP: ₹159999.00 (Incl. of all taxes)    4.

In [6]:
df2 = df.copy()

In [7]:
df2.to_csv('raw_smartphones.csv')

In [8]:
df.sample(10)

,name,price,original_price,rating,reviews,color,storage
32,Galaxy A07 (4 GB Memory),₹10999.00,MRP: ₹12999.00 (Incl. of all taxes),None,None,Green,64 GB
51,Galaxy A36 5G Certified Re-Newed (8 GB Memory),₹23249.00,MRP: ₹39499.00 (Incl. of all taxes),3.8,259,Awesome Lavender,128 GB
36,Galaxy F36 5G (8 GB Memory),₹23999.00,MRP: ₹29999.00 (Incl. of all taxes),None,None,Luxe Violet,128 GB
52,Galaxy A56 5G Certified Re-Newed (8 GB Memory),₹31499.00,MRP: ₹54999.00 (Incl. of all taxes),4,187,Awesome Olive,256 GB
4,Galaxy S26+ (Special Colour),₹94999.00,MRP: ₹119999.00 (Incl. of all taxes),4.8,"1,918",Pinkgold,256 GB
42,Galaxy A56 5G (8 GB Memory),₹44999.00,MRP: ₹54999.00 (Incl. of all taxes),4.4,892,Awesome Olive,256 GB
16,Galaxy S25 Ultra (Special Colour),₹99999.00,MRP: ₹129999.00 (Incl. of all taxes),4.8,"9,999+",Titanium Jetblack,256 GB
81,Galaxy A06 (4 GB Memory),None,None,None,None,Light Blue,64 GB
80,Galaxy M15 5G Prime (8 GB Memory),None,None,5,1,Dark Blue,128 GB
2,Galaxy S26,₹99999.00,MRP: ₹107999.00 (Incl. of all taxes),4.8,"2,231",Cobalt Violet,512 GB


In [9]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   name            104 non-null    object
 1   price           60 non-null     object
 2   original_price  60 non-null     object
 3   rating          55 non-null     object
 4   reviews         55 non-null     object
 5   color           103 non-null    object
 6   storage         103 non-null    object
dtypes: object(7)
memory usage: 5.8+ KB


# issues
## table 1 
### dirty data
- data type of price col is object `validity`
- price contain nan values `Completeness`
- original price column conatin many unwanted things  `validity`
- data type of review columns is object `validity`
- reviews col contain nan values `coompleteness`
- data type of rating columns is object `validity`
- rating col contain nan values `Completeness`
- color col contain nan values `Completeness`
- storage col contain nan values `Completeness`
- storage col is in str instance of int  `validity`1
- none used to represent nan `validity`
### messy data
- in col name contain memory also
- two price columns 

In [10]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   name            104 non-null    object
 1   price           60 non-null     object
 2   original_price  60 non-null     object
 3   rating          55 non-null     object
 4   reviews         55 non-null     object
 5   color           103 non-null    object
 6   storage         103 non-null    object
dtypes: object(7)
memory usage: 5.8+ KB


In [11]:
import numpy as np

df2 = df2.replace({None: np.nan , 'NaN':np.nan})
df2.drop(columns='original_price',inplace =True)

In [12]:
df2.dropna(subset='price',inplace=True,ignore_index=True)


In [13]:
df2.fillna(value={'reviews': 0,'rating':0},inplace=True)

In [14]:
df2.dropna(subset='name',inplace=True,ignore_index=True)


In [15]:
df2['price'] = (
    df['price']
    .str.replace('₹', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

In [16]:
df3 = df2.rename(columns={'price': 'price(inr)'}, inplace=True)

In [17]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        60 non-null     object 
 1   price(inr)  55 non-null     float64
 2   rating      60 non-null     object 
 3   reviews     60 non-null     object 
 4   color       60 non-null     object 
 5   storage     60 non-null     object 
dtypes: float64(1), object(5)
memory usage: 2.9+ KB


In [18]:
df2['rating'] =df2['rating'].astype(float)

In [19]:
df['reviews'] = (
    df['reviews'].astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('+', '', regex=False)
)



In [20]:
df['reviews'] = pd.to_numeric(df['reviews'], errors='coerce').astype('Int64')

In [21]:
print(df2.columns.tolist())

['name', 'price(inr)', 'rating', 'reviews', 'color', 'storage']


In [22]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        60 non-null     object 
 1   price(inr)  55 non-null     float64
 2   rating      60 non-null     float64
 3   reviews     60 non-null     object 
 4   color       60 non-null     object 
 5   storage     60 non-null     object 
dtypes: float64(2), object(4)
memory usage: 2.9+ KB


In [23]:
df2['storage']=df2['storage'].str.replace('GB', '', regex=False).str.replace('TB', '', regex=False).astype("int64")

In [24]:
df3 = df2.rename(columns={'storage': 'storage(GB)'}, inplace=True)

In [29]:
df2[df2['storage(GB)']<120] = 1024

In [30]:
df2[df2['storage(GB)']>500]

,name,price(inr),rating,reviews,color,storage(GB)
2,Galaxy S26,99999.0,4.8,"2,231",Cobalt Violet,512
3,Galaxy S26 Ultra (Special Colour),150999.0,4.8,"9,150",Pinkgold,512
5,Galaxy S26 (Special Colour),99999.0,4.8,"2,231",Pinkgold,512
32,1024,1024.0,1024.0,1024,1024,1024
33,1024,1024.0,1024.0,1024,1024,1024
45,1024,1024.0,1024.0,1024,1024,1024
47,1024,1024.0,1024.0,1024,1024,1024


In [31]:
df2.sample(20)

,name,price(inr),rating,reviews,color,storage(GB)
25,Galaxy M17 5G (4 GB Memory),16999.0,0.0,0,Moonlight Silver,128
5,Galaxy S26 (Special Colour),99999.0,4.8,"2,231",Pinkgold,512
24,Galaxy M36 5G (6 GB Memory),21999.0,0.0,0,Serene Green,128
53,Galaxy S25 Edge,32749.0,4.7,817,Titanium Silver,256
57,Galaxy M06 5G (4 GB Memory),NaN,0.0,0,Blazing Black,128
38,Galaxy Z Flip7,109999.0,4.6,"2,230",Blue Shadow,256
44,Galaxy A06 5G (6 GB Memory),17999.0,0.0,0,Light Green,128
15,Galaxy F70e 5G (4 GB Memory),15499.0,0.0,0,Limelight Green,128
19,Galaxy A36 5G (8 GB Memory),33999.0,4.2,854,Awesome Lavender,128
36,Galaxy F36 5G (8 GB Memory),23999.0,0.0,0,Luxe Violet,128


In [32]:
df2.dropna(inplace=True,ignore_index=True)

In [33]:
print(df2.isna().sum())

name           0
price(inr)     0
rating         0
reviews        0
color          0
storage(GB)    0
dtype: int64


In [34]:
df2['name'] = df2['name'].str.replace(
    r'\s*\(\d+\s*GB Memory\)',
    '',
    regex=True
)

In [35]:
df2['price(inr)']=df2['price(inr)'].astype("int64")

In [36]:
df2 = df2.loc[:, ~df2.columns.duplicated(keep='last')]

In [37]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   name         52 non-null     object 
 1   price(inr)   56 non-null     int64  
 2   rating       56 non-null     float64
 3   reviews      56 non-null     object 
 4   color        56 non-null     object 
 5   storage(GB)  56 non-null     int64  
dtypes: float64(1), int64(2), object(3)
memory usage: 2.8+ KB


In [38]:
df2

,name,price(inr),rating,reviews,color,storage(GB)
0,Galaxy S26 Ultra,130999,4.8,"9,150",Cobalt Violet,256
1,Galaxy S26+,94999,4.8,"1,918",Black,256
2,Galaxy S26,99999,4.8,"2,231",Cobalt Violet,512
3,Galaxy S26 Ultra (Special Colour),150999,4.8,"9,150",Pinkgold,512
4,Galaxy S26+ (Special Colour),94999,4.8,"1,918",Pinkgold,256
5,Galaxy S26 (Special Colour),99999,4.8,"2,231",Pinkgold,512
6,Galaxy A57 5G,57999,0.0,0,Awesome Lilac,256
7,Galaxy A57 5G,53999,0.0,0,Awesome Navy,256
8,Galaxy A37 5G,52999,0.0,0,Awesome Lavender,256
9,Galaxy A37 5G,41999,0.0,0,Awesome Lavender,128


In [39]:
df2.drop_duplicates(subset='name',inplace=True,ignore_index=True)

In [40]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   name         37 non-null     object 
 1   price(inr)   38 non-null     int64  
 2   rating       38 non-null     float64
 3   reviews      38 non-null     object 
 4   color        38 non-null     object 
 5   storage(GB)  38 non-null     int64  
dtypes: float64(1), int64(2), object(3)
memory usage: 1.9+ KB


In [44]:
df2.drop(index=24,inplace=True)


In [45]:
df2.to_csv('smartphones_cleaned.csv')